In [27]:
import os
from tifffile import imread, imwrite
import shutil
import random
from tqdm import tqdm
from skimage.measure import label
from skimage.segmentation import clear_border
from cellpose import io, models, train
from utils import list_images, read_image, fill_label_holes
import napari

#Load pre-trained Cellpose-SAM
model = models.CellposeModel(gpu=True)

In [28]:
# Copy the path to the folder containing your images in between the quotation marks
data_folder = "./training_data"

# Define brightfield channel position to extract from the image
brightfield_channel = 0

In [29]:
# Create a folder to hold the _img and _masks pairs for CellposeSAM fine-tuning
curated_data_folder = "curated_training_data"
if not os.path.exists(curated_data_folder):
    os.makedirs(curated_data_folder)

# If the path is correct you should see a list of the first 10 images in your folder down below
images = list_images(data_folder, format="nd2")
images[:10]

['training_data\\260324_SK_SK0064_Exp1_WellE2_after4h_Pos003.nd2',
 'training_data\\260324_SK_SK0064_Exp1_WellE3_after4h_Pos006.nd2',
 'training_data\\260324_SK_SK0064_Exp1_WellE4_after4h_Pos013.nd2']

In [30]:
for img_filepath in tqdm(images):

    # Read the .nd2 file and extract its corresponding filename
    img, filename = read_image(img_filepath)

    # Extract the brightfield channel
    brightfield_channel = img[0]

    # Save brightfield channel with _img suffix for training
    imwrite(f"./curated_training_data/{filename}_img.tif",brightfield_channel)

    # Load the corresponding labels
    labels = imread(f"./training_data/{filename}_cell_labels.tif")
    # Relabel so there are not repeated label values from the annotation process
    labels = label(labels)
    # Fix small holes derived from the manual annotation process
    labels = fill_label_holes(labels)
    # Remove labels touching the image edge
    labels_clean = clear_border(labels)

    # Save fixed labels with _mask suffix for training
    imwrite(f"./curated_training_data/{filename}_masks.tif", labels)

  0%|          | 0/3 [00:00<?, ?it/s]


Image analyzed: 260324_SK_SK0064_Exp1_WellE2_after4h_Pos003


 33%|███▎      | 1/3 [00:01<00:02,  1.45s/it]


Image analyzed: 260324_SK_SK0064_Exp1_WellE3_after4h_Pos006


 67%|██████▋   | 2/3 [00:02<00:01,  1.38s/it]


Image analyzed: 260324_SK_SK0064_Exp1_WellE4_after4h_Pos013


100%|██████████| 3/3 [00:04<00:00,  1.47s/it]


In [31]:
# Train/test split
# Set random seed for reproducibility
random.seed(42)

curated_data_folder = "curated_training_data"
train_dir = "train_dir"
test_dir = "test_dir"

# List all unique stem names in curated_training_data (without _img/_masks/tif)
stems = set()
for f in os.listdir(curated_data_folder):
    if f.endswith("_img.tif"):
        stem = f[:-8]  # remove _img.tif
        # Ensure both image and mask are present for this stem
        if (os.path.exists(os.path.join(curated_data_folder, f"{stem}_img.tif")) and
            os.path.exists(os.path.join(curated_data_folder, f"{stem}_masks.tif"))):
            stems.add(stem)

stems = list(stems)
random.shuffle(stems)
split_idx = int(0.8 * len(stems))
train_stems = stems[:split_idx]
test_stems = stems[split_idx:]

# Create output folders if not exists
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Helper to move files
def move_pair(stems_subset, dest_dir):
    for stem in stems_subset:
        img_src = os.path.join(curated_data_folder, f"{stem}_img.tif")
        mask_src = os.path.join(curated_data_folder, f"{stem}_masks.tif")
        img_dest = os.path.join(dest_dir, f"{stem}_img.tif")
        mask_dest = os.path.join(dest_dir, f"{stem}_masks.tif")
        shutil.move(img_src, img_dest)
        shutil.move(mask_src, mask_dest)

move_pair(train_stems, train_dir)
move_pair(test_stems, test_dir)

# Remove the temporary curated_training_data folder
shutil.rmtree(curated_data_folder)

In [ ]:
# CellposeSAM finetuning
io.logger_setup()

# Load the train and test data
output = io.load_train_test_data(train_dir, test_dir, image_filter="_img",
                                mask_filter="_masks", look_one_level_down=False)

# Unpack the tuple                                
images, labels, image_names, test_images, test_labels, image_names_test = output

# Start training
model_path, train_losses, test_losses = train.train_seg(model.net,
                            train_data=images, train_labels=labels,
                            test_data=test_images, test_labels=test_labels,
                            weight_decay=1e-4, learning_rate=1e-4, # decrease weight decay and increase learning rate (small dataset)
                            n_epochs=100, model_name="CPSAM_shirik_ft") # decrease epochs to avoid overfitting, LR collapses around epoch 50-60

creating new log file
2026-05-21 10:40:41,151 [INFO] WRITING LOG OUTPUT TO C:\Users\adiez_cmic\.cellpose\run.log
2026-05-21 10:40:41,152 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0
2026-05-21 10:40:41,153 [INFO] not all flows are present, running flow generation for all images
2026-05-21 10:40:41,287 [INFO] 2 / 2 images in train_dir folder have labels
2026-05-21 10:40:41,287 [INFO] not all flows are present, running flow generation for all images
2026-05-21 10:40:41,352 [INFO] 1 / 1 images in test_dir folder have labels
2026-05-21 10:40:41,371 [INFO] computing flows for labels


100%|██████████| 2/2 [00:52<00:00, 26.18s/it]

2026-05-21 10:41:33,839 [INFO] computing flows for labels


2026-05-21 10:41:55,550 [INFO] >>> computing diameters


100%|██████████| 1/1 [00:00<00:00, 12.91it/s]

2026-05-21 10:41:55,735 [INFO] >>> normalizing {'lowhigh': None, 'percentile': None, 'normalize': True, 'norm3D': True, 'sharpen_radius': 0, 'smooth_radius': 0, 'tile_norm_blocksize': 0, 'tile_norm_smooth3D': 1, 'invert': False}


2026-05-21 10:41:56,154 [INFO] >>> n_epochs=100, n_train=2, n_test=1
2026-05-21 10:41:56,154 [INFO] >>> AdamW, learning_rate=0.00010, weight_decay=0.00010
2026-05-21 10:41:56,158 [INFO] >>> saving model to c:\Users\adiez_cmic\github_repos\shirik_RCD_analysis\models\CPSAM_shirik_ft
2026-05-21 10:41:56,825 [INFO] 0, train_loss=1.8125, test_loss=6.4062, LR=0.000000, time 0.67s
2026-05-21 10:41:59,359 [INFO] 5, train_loss=1.4995, test_loss=4.3438, LR=0.000056, time 3.20s
2026-05-21 10:42:01,980 [INFO] 10, train_loss=2.2109, test_loss=3.8125, LR=0.000100, time 5.83s
2026-05-21 10:42:07,071 [INFO] 20, train_loss=1.9994, test_loss=3.9375, LR=0.000100, time 10.92s
2026-05-21 10:42:12,862 [INFO] 30, train_loss=2.0355, test_loss=3.5625, LR=0.000100, time 16.71s
2026-05-21 10:42:18,195 [INFO] 40, train_loss=1.7560, test_loss=3.4688, LR=0.000100, time 22.04s
2026-05-21 10:42:23,327 [INFO] 50, train_loss=1.4811, test_loss=4.3125, LR=0.000050, time 27.17s
2026-05-21 10:42:29,247 [INFO] 60, train_los

In [ ]:
# Compare original CellposeSAM with fine-tuned one
input_brightfield = imread("./test_dir/260324_SK_SK0064_Exp1_WellE2_after4h_Pos003_img.tif")
annotations = imread("./test_dir/260324_SK_SK0064_Exp1_WellE2_after4h_Pos003_masks.tif")

#Load pre-trained Cellpose-SAM
model = models.CellposeModel(gpu=True)

# Predict labels using original CellposeSAM
og_cell_labels, flows, styles = model.eval(input_brightfield, niter=1000)

# Load fine-tuned Cellpose-SAM model from ./models directory
ft_model = models.CellposeModel(gpu=True, pretrained_model="./models/CPSAM_shirik_ft")

# Predict labels using fine-tuned CellposeSAM
ft_cell_labels, flows, styles = ft_model.eval(input_brightfield, niter=1000)


2026-05-21 11:00:16,583 [INFO] ** TORCH CUDA version installed and working. **
2026-05-21 11:00:16,583 [INFO] >>>> using GPU (CUDA)
2026-05-21 11:00:17,892 [INFO] >>>> loading model C:\Users\adiez_cmic\.cellpose\models\cpsam
2026-05-21 11:00:37,263 [INFO] ** TORCH CUDA version installed and working. **
2026-05-21 11:00:37,263 [INFO] >>>> using GPU (CUDA)
2026-05-21 11:00:38,331 [INFO] >>>> loading model ./models/CPSAM_shirik_ft


In [44]:
# Show results and compare vs manual annotation
viewer = napari.Viewer(ndisplay=2)
viewer.add_image(input_brightfield)
viewer.add_labels(annotations)
viewer.add_labels(og_cell_labels, name="og CPSAM labels")
viewer.add_labels(ft_cell_labels, name="fine-tuned CPSAM labels")

<Labels layer 'fine-tuned CPSAM labels' at 0x2b9c13cd360>